In [ ]:
import numpy as np
import pandas as pd
import importlib

import src.utils.state as state
import src.utils.plottings as plotting
import src.utils.scenarioCalculator as scenarioCalculator
import src.utils.dataConverter as dataConverter
import src.utils.config as config
import src.utils.models as models
from src.utils.config import GlobalConfig
from src.utils.scenarios import Scenarios
from src.utils.models import Scenario

importlib.reload(plotting)
importlib.reload(state)
importlib.reload(scenarioCalculator)
importlib.reload(dataConverter)
importlib.reload(models)
importlib.reload(config)


# Preliminaries

## 1. Reading data

In [ ]:
df_cases = pd.read_excel("../resources/weekly_varicella_hun.xlsx", sheet_name="weekly_cases")
df_age_structured_cases = pd.read_excel("../resources/age_structured_vzv_hun.xlsx", sheet_name="varicella")
df_susceptibles = pd.read_excel("../resources/number_of_susceptibles_hun.xlsx", sheet_name="s0")
df_population = pd.read_excel("../resources/age_structured_population.xlsx", sheet_name="population")
df_births = pd.read_excel("../resources/births_hun.xlsx", sheet_name="births")
df_daily_births = pd.read_excel("../resources/births_hun.xlsx", sheet_name="daily", header=None)
df_deaths = pd.read_excel("../resources/age_structured_deaths_hun.xlsx", sheet_name="deaths")
df_death_proportions = pd.read_excel("../resources/age_structured_deaths_hun.xlsx", sheet_name="proportions")
df_vaccines = pd.read_excel("../resources/vaccine_coverage_hun.xlsx", sheet_name="vaccines")
df_contacts = pd.read_excel("../resources/contact_mtx_hun.xlsx", sheet_name="contacts", index_col=0)

## 2. Convert dates and set indices

### 2.1 Varicella data

#### 2.1.1 Weekly cases

In [ ]:
weekly_cases = dataConverter.convert_df_cases_to_weekly_cases(df_cases)
state.set_weekly_cases(weekly_cases)

# fill missing values
full_index = dataConverter.create_full_index_date_range(weekly_cases.index.max())
state.set_full_index(full_index)

state.set_weekly_cases_related_values()

#### 2.1.2 Age-structured data

In [ ]:
state.set_age_groups_and_nr_age_groups(df_age_structured_cases.iloc[:, 0].astype(str).tolist())
state.set_years(df_age_structured_cases.columns[1:].astype(int).tolist())

annual_cases = dataConverter.convert_df_age_structured_cases(df_age_structured_cases)
state.set_annual_cases(annual_cases)

In [ ]:
case_matrix = dataConverter.create_case_matrix(annual_cases, state.WEEKLY_CASES_FILLED, state.WEEKLY_CASES_FULL)
state.set_cases_matrix_and_global_min_max(case_matrix)

In [ ]:
state.set_age_structured_data_mask(state.WEEKLY_CASES_FILLED.index.year <= max(state.YEARS))
state.set_age_structured_data_index(state.WEEKLY_CASES_FILLED.loc[state.AGE_STRUCTURED_DATA_MASK].index)
i = case_matrix
state.set_nr_timesteps(len(state.WEEKLY_CASES_FILLED[state.AGE_STRUCTURED_DATA_MASK]))

### 2.2 Birth and death data; vaccination coverage

In [ ]:
state.set_weekly_index_related_values(state.WEEKLY_CASES_FILLED[state.AGE_STRUCTURED_DATA_MASK].index)

annual_deaths = dataConverter.convert_annual_deaths(df_deaths)
death_matrix = dataConverter.create_weekly_death_matrix(annual_deaths, state.WEEKLY_INDEX)
#death_matrix = dataConverter.create_weekly_death_matrix(annual_deaths, state.WEEKLY_CASES_FULL)
state.set_death_matrix(death_matrix)

#deaths_prop_matrix = dataConverter.convert_weekly_death_prop_data_to_death_prop_mtx(df_death_proportions)
#state.set_death_matrix(deaths_prop_matrix)

In [ ]:
#birth_mtx = dataConverter.convert_annual_birth_data_to_birth_mtx(df_births)
birth_mtx = dataConverter.convert_daily_birth_data_to_birth_mtx(df_daily_births)
state.set_birth_matrix(birth_mtx)
weekly_v1 = dataConverter.convert_annual_v1_data_to_weekly_v1(df_vaccines)
state.set_weekly_v1(weekly_v1)
weekly_v2 = dataConverter.convert_annual_v2_data_to_weekly_v2(df_vaccines)
state.set_weekly_v2(weekly_v2)

### 2.3 Initial values

#### 2.3.1 Number of susceptibles

In [ ]:
S0_vector = dataConverter.convert_data_to_s0(df_susceptibles)
state.set_s0_vector(S0_vector)

#### 2.3.2 Population size

In [ ]:
N0_vector = dataConverter.convert_data_to_n0(df_population)
state.set_n0_vector(N0_vector)

### 2.4 Contact matrix

In [ ]:
contacts0 = dataConverter.convert_data_to_contact_matrix(df_contacts)
state.set_contacts0(contacts0)

# Baseline scenario

## 3. Calculate number of susceptibles

In [ ]:
ALL_SCENARIOS: list[Scenario] = []

In [ ]:
state.set_weekly_v1_age_structured()
state.set_weekly_v2_age_structured()

scenarioCalculator.calculate_number_of_susceptible_cases_in_baseline(True)
ALL_SCENARIOS.append(state.BASELINE_SCENARIO)

## 4. Estimate R_t using renewal equation model

In [ ]:
scenarioCalculator.calculate_remainders(True)
plotting.plot_vector_with_index(state.RHO, state.AGE_STRUCTURED_DATA_INDEX, "Rho", "Date", "rho")

## 5. Plot results for R_t

In [ ]:
plotting.plot_remainders()

## 6. Plot results for the number of incidences

In [ ]:
scenarioCalculator.calculate_number_of_incidences_based_on_remainders(True)

plotting.plot_model_actual_scat_comparison(state.BASELINE_SCENARIO)
plotting.plot_annual_cases(state.BASELINE_SCENARIO)

In [ ]:
plotting.plot_model_results_age_range_in_range(state.BASELINE_SCENARIO.V1, 0, 2, state.BASELINE_SCENARIO.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(state.BASELINE_SCENARIO.I, 0, 4, state.BASELINE_SCENARIO.name)
plotting.plot_model_results_age_range_in_range(state.BASELINE_SCENARIO.I, 5, 9, state.BASELINE_SCENARIO.name)
plotting.plot_model_results_age_range_in_range(state.BASELINE_SCENARIO.I, 10, 12, state.BASELINE_SCENARIO.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(state.BASELINE_SCENARIO.I, 2, 3, state.BASELINE_SCENARIO.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(state.BASELINE_SCENARIO.S, 0, 4, state.BASELINE_SCENARIO.name)
plotting.plot_model_results_age_range_in_range(state.BASELINE_SCENARIO.S, 5, 9, state.BASELINE_SCENARIO.name)
plotting.plot_model_results_age_range_in_range(state.BASELINE_SCENARIO.S, 10, 12, state.BASELINE_SCENARIO.name)

In [ ]:
plotting.plot_scat_age_range_in_range(state.BASELINE_SCENARIO.I, state.AGE_STRUCTURED_DATA_INDEX, 0, 4, state.CASES_MATRIX, state.AGE_STRUCTURED_DATA_MASK)
plotting.plot_scat_age_range_in_range(state.BASELINE_SCENARIO.I, state.AGE_STRUCTURED_DATA_INDEX, 5, 9, state.CASES_MATRIX, state.AGE_STRUCTURED_DATA_MASK)
plotting.plot_scat_age_range_in_range(state.BASELINE_SCENARIO.I, state.AGE_STRUCTURED_DATA_INDEX, 10, 13, state.CASES_MATRIX, state.AGE_STRUCTURED_DATA_MASK)

# Change vaccination level

## 7. Incidences without vaccination

In [ ]:
scenario_no_vacc = scenarioCalculator.calculate_number_of_cases_for_scenario(
    Scenarios.NO_VACCINATION, 0, GlobalConfig.START_OF_VACCINATION, True)
ALL_SCENARIOS.append(scenario_no_vacc)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_no_vacc)
plotting.plot_model_results_age_range_in_range(scenario_no_vacc.I, 0, 4, scenario_no_vacc.name)
plotting.plot_model_results_age_range_in_range(scenario_no_vacc.I, 5, 9, scenario_no_vacc.name)
plotting.plot_model_results_age_range_in_range(scenario_no_vacc.I, 10, 12, scenario_no_vacc.name)

In [ ]:
plotting.plot_annual_annual_number_of_cases_for_age_groups(scenario_no_vacc.I, 0, 4, scenario_no_vacc.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(scenario_no_vacc.weekly_i_series, scenario_no_vacc.name)

## 8. Vaccination level = 0.75

In [ ]:
scenario_vacc_level_75: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(
    Scenarios.VACC_LEVEL_75, 0.75, GlobalConfig.START_OF_VACCINATION, True)
ALL_SCENARIOS.append(scenario_vacc_level_75)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_vacc_level_75)
plotting.plot_model_results_age_range_in_range(scenario_vacc_level_75.I, 0, 4, scenario_vacc_level_75.name)
plotting.plot_model_results_age_range_in_range(scenario_vacc_level_75.I, 5, 9, scenario_vacc_level_75.name)
plotting.plot_model_results_age_range_in_range(scenario_vacc_level_75.I, 10, 12, scenario_vacc_level_75.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(scenario_vacc_level_75.weekly_i_series, scenario_vacc_level_75.name)

## 9. Vaccination level = 0.5

In [ ]:
scenario_vacc_level_50: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(
    Scenarios.VACC_LEVEL_50, 0.50, GlobalConfig.START_OF_VACCINATION, True)
ALL_SCENARIOS.append(scenario_vacc_level_50)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_vacc_level_50)
plotting.plot_model_results_age_range_in_range(scenario_vacc_level_50.I, 0, 4, scenario_vacc_level_50.name)
plotting.plot_model_results_age_range_in_range(scenario_vacc_level_50.I, 5, 9, scenario_vacc_level_50.name)
plotting.plot_model_results_age_range_in_range(scenario_vacc_level_50.I, 10, 12, scenario_vacc_level_50.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(scenario_vacc_level_50.weekly_i_series, scenario_vacc_level_50.name)

## 10. Vaccination level = 0.25

In [ ]:
scenario_vacc_level_25: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(
    Scenarios.VACC_LEVEL_25, 0.25, GlobalConfig.START_OF_VACCINATION, True)
ALL_SCENARIOS.append(scenario_vacc_level_25)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_vacc_level_25)
plotting.plot_model_results_age_range_in_range(scenario_vacc_level_25.I, 0, 4, scenario_vacc_level_25.name)
plotting.plot_model_results_age_range_in_range(scenario_vacc_level_25.I, 5, 9, scenario_vacc_level_25.name)
plotting.plot_model_results_age_range_in_range(scenario_vacc_level_25.I, 10, 12, scenario_vacc_level_25.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(scenario_vacc_level_25.weekly_i_series, scenario_vacc_level_25.name)

# Starting vaccination earlier

## 11. Start vaccination 1 year earlier (2018.09.01)

In [ ]:
scenario_starting_0_year_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(
    "test starting 0 years minus", 1, GlobalConfig.START_OF_VACCINATION, True)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_starting_0_year_minus)
plotting.plot_model_results_age_range_in_range(scenario_starting_0_year_minus.I, 0, 4, scenario_starting_0_year_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_starting_0_year_minus.I, 5, 9, scenario_starting_0_year_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_starting_0_year_minus.I, 10, 12, scenario_starting_0_year_minus.name)

In [ ]:
scenario_1_year_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(
    Scenarios.STARTING_1_YEAR_MINUS, 1, GlobalConfig.START_OF_VACCINATION + pd.DateOffset(years=-1), True)
ALL_SCENARIOS.append(scenario_1_year_minus)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_1_year_minus)
plotting.plot_model_results_age_range_in_range(scenario_1_year_minus.I, 0, 4, scenario_1_year_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_1_year_minus.I, 5, 9, scenario_1_year_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_1_year_minus.I, 10, 12, scenario_1_year_minus.name)

In [ ]:
test_starting_time = pd.Timestamp(2007,9,1)
scenario_n_months_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(
    str(test_starting_time), 0.25, test_starting_time, True)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_n_months_minus)
plotting.plot_model_results_age_range_in_range(scenario_n_months_minus.I, 0, 4, scenario_n_months_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_n_months_minus.I, 5, 9, scenario_n_months_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_n_months_minus.I, 10, 12, scenario_n_months_minus.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(scenario_n_months_minus.weekly_i_series, scenario_n_months_minus.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(scenario_n_months_minus.V1, 0, 1, scenario_n_months_minus.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(scenario_1_year_minus.weekly_i_series, scenario_starting_0_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(scenario_1_year_minus.I, 0, 4, scenario_starting_0_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(scenario_1_year_minus.I, 5, 9, scenario_starting_0_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(scenario_1_year_minus.I, 10, 12, scenario_starting_0_year_minus.name)

## 12. Start vaccination 2 years earlier (2017.09.01)

In [ ]:
scenario_2_years_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(
    Scenarios.STARTING_2_YEAR_MINUS, 1, GlobalConfig.START_OF_VACCINATION + pd.DateOffset(years=-2), True)
ALL_SCENARIOS.append(scenario_2_years_minus)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_2_years_minus)
plotting.plot_model_results_age_range_in_range(scenario_2_years_minus.I, 0, 4, scenario_2_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_2_years_minus.I, 5, 9, scenario_2_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_2_years_minus.I, 10, 12, scenario_2_years_minus.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(scenario_1_year_minus.V1, 0, 2, scenario_1_year_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_2_years_minus.V1, 0, 2, scenario_2_years_minus.name)

In [ ]:
plotting.plot_all_susceptibles(scenario_2_years_minus)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(scenario_2_years_minus.weekly_i_series, scenario_2_years_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(scenario_2_years_minus.I, 0, 4, scenario_2_years_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(scenario_2_years_minus.I, 5, 9, scenario_2_years_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(scenario_2_years_minus.I, 10, 12, scenario_2_years_minus.name)

## 13. Start vaccination 3 years earlier (2016.09.01)

In [ ]:
scenario_3_years_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(
    Scenarios.STARTING_3_YEAR_MINUS, 1, GlobalConfig.START_OF_VACCINATION + pd.DateOffset(years=-3), True)
ALL_SCENARIOS.append(scenario_3_years_minus)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_3_years_minus)
plotting.plot_model_results_age_range_in_range(scenario_3_years_minus.I, 0, 4, scenario_3_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_3_years_minus.I, 5, 9, scenario_3_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_3_years_minus.I, 10, 12, scenario_3_years_minus.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(scenario_3_years_minus.V1, 0, 2, scenario_3_years_minus.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(scenario_3_years_minus.weekly_i_series, scenario_3_years_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(scenario_3_years_minus.I, 0, 4, scenario_3_years_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(scenario_3_years_minus.I, 5, 9, scenario_3_years_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(scenario_3_years_minus.I, 10, 12, scenario_3_years_minus.name)

## 14. Start vaccination 5 years earlier (2014.09.01)

In [ ]:
scenario_5_years_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(
    Scenarios.STARTING_5_YEAR_MINUS, 1, GlobalConfig.START_OF_VACCINATION + pd.DateOffset(years=-5), True)
ALL_SCENARIOS.append(scenario_5_years_minus)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_5_years_minus)
plotting.plot_model_results_age_range_in_range(scenario_5_years_minus.I, 0, 4, scenario_5_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_5_years_minus.I, 5, 9, scenario_5_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_5_years_minus.I, 10, 12, scenario_5_years_minus.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(scenario_5_years_minus.V1, 1, 1, scenario_5_years_minus.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(scenario_5_years_minus.S, 0, 4, scenario_5_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_5_years_minus.S, 5, 9, scenario_5_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_5_years_minus.S, 10, 12, scenario_5_years_minus.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(scenario_5_years_minus.weekly_i_series, scenario_5_years_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(scenario_5_years_minus.I, 0, 4, scenario_5_years_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(scenario_5_years_minus.I, 5, 9, scenario_5_years_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(scenario_5_years_minus.I, 10, 12, scenario_5_years_minus.name)

## 15. Starting the vaccination 8 years earlier (2011.09.01)

In [ ]:
scenario_8_years_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(
    Scenarios.STARTING_8_YEAR_MINUS, 1, GlobalConfig.START_OF_VACCINATION + pd.DateOffset(years=-8), True)
ALL_SCENARIOS.append(scenario_8_years_minus)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_8_years_minus)
plotting.plot_model_results_age_range_in_range(scenario_8_years_minus.I, 0, 4, scenario_8_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_8_years_minus.I, 5, 9, scenario_8_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_8_years_minus.I, 10, 12, scenario_8_years_minus.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(scenario_8_years_minus.weekly_i_series, scenario_8_years_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(scenario_8_years_minus.I, 0, 4, scenario_8_years_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(scenario_8_years_minus.I, 5, 9, scenario_8_years_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(scenario_8_years_minus.I, 10, 12, scenario_8_years_minus.name)

## 16. Starting the vaccination 13 years earlier (2006.09.01)

In [ ]:
scenario_13_years_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(
    Scenarios.STARTING_13_YEAR_MINUS, 1, GlobalConfig.START_OF_VACCINATION + pd.DateOffset(years=-13), True)
ALL_SCENARIOS.append(scenario_13_years_minus)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_13_years_minus)
plotting.plot_model_results_age_range_in_range(scenario_13_years_minus.I, 0, 4, scenario_13_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_13_years_minus.I, 5, 9, scenario_13_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_13_years_minus.I, 10, 12, scenario_13_years_minus.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(scenario_13_years_minus.weekly_i_series, scenario_13_years_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(scenario_13_years_minus.I, 0, 4, scenario_13_years_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(scenario_13_years_minus.I, 5, 9, scenario_13_years_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(scenario_13_years_minus.I, 10, 12, scenario_13_years_minus.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(scenario_13_years_minus.V1, 0, 1, scenario_13_years_minus.name)

In [ ]:
plotting.plot_cumulative_i_and_v(scenario_13_years_minus, pd.Timestamp(2007,1,1))

# Summaries

## 17. Cumulative plots

In [ ]:
mask = np.ones(len(state.AGE_STRUCTURED_DATA_INDEX), dtype=bool)
plotting.plot_cumulative_cases(ALL_SCENARIOS, state.AGE_STRUCTURED_DATA_INDEX, mask)

In [ ]:
index_after_2016_1_1 = state.WEEKLY_INDEX[state.AFTER_2016_1_1_MASK]
plotting.plot_cumulative_cases(ALL_SCENARIOS, index_after_2016_1_1, state.AFTER_2016_1_1_MASK)

In [ ]:
AFTER_2020_1_1_MASK = (state.WEEKLY_INDEX > pd.Timestamp(2020, 1, 1)) & (state.WEEKLY_INDEX < pd.Timestamp(2021, 1, 1))
index_after_2020_1_1 = state.WEEKLY_INDEX[AFTER_2020_1_1_MASK]
plotting.plot_cumulative_cases(ALL_SCENARIOS, index_after_2020_1_1, AFTER_2020_1_1_MASK)

## 18. Heatmaps

In [ ]:
for scenario in ALL_SCENARIOS:
    df_scenario = pd.DataFrame(scenario.I, index=state.AGE_GROUPS, columns=state.WEEKLY_INDEX)
    plotting.plot_heatmap(df_scenario, "The change of varicella age distribution - " + scenario.name)

# Without Covid

## 14. Calculate mean R_t

In [ ]:
def calculate_mean_during_covid(data: np.ndarray[float], index: pd.Series) -> np.ndarray[float]:
    df: pd.DataFrame = pd.DataFrame(data.T, index=index, columns=state.AGE_GROUPS)
    timerange_for_mean = ((df.index > config.GlobalConfig.START_OF_VACCINATION + pd.DateOffset(years=-5))
                         & (df.index < config.GlobalConfig.START_OF_VACCINATION))
    timerange_for_mean = (((df.index > pd.Timestamp(2019,1,1) + pd.DateOffset(years=-2)) & (df.index < pd.Timestamp(2019,1,1))) |
                          ((df.index > pd.Timestamp("2020-03-16")) & (df.index < pd.Timestamp("2020-03-16") + pd.DateOffset(years=2))))
    mean_data: pd.DataFrame = df[timerange_for_mean].groupby(df.index[timerange_for_mean].map(lambda i: i.week)).mean()
    no_cov_data: np.ndarray = np.zeros((len(state.AGE_GROUPS), len(state.WEEKLY_INDEX)))
    min_t = len(state.WEEKLY_INDEX) - len(index)
    max_t = len(state.WEEKLY_INDEX)
    for t in range(0, len(state.WEEKLY_INDEX)):
    #for t in range(min_t, max_t):
        #no_cov_data[:, t] = data[:, t - min_t]
        no_cov_data[:, t] = data[:, t]
        if (state.WEEKLY_INDEX[t] > pd.Timestamp("2020-03-16")) and (state.WEEKLY_INDEX[t] < pd.Timestamp("2021-09-01")):
            week_of_year: int = state.WEEKLY_INDEX[t].week
            if week_of_year > len(mean_data):
                no_cov_data[:, t] = mean_data.loc[week_of_year-1][:]
            else:
                no_cov_data[:, t] = mean_data.loc[week_of_year][:]
    return no_cov_data

In [ ]:
#mean_remainders: np.ndarray[float] = calculate_mean_during_covid(state.REMAINDERS[:,3:], state.WEEKLY_INDEX[3:])
#mean_remainders[:,:3] = state.REMAINDERS[:,:3]
mean_remainders: np.ndarray[float] = calculate_mean_during_covid(state.REMAINDERS, state.WEEKLY_INDEX)
#state.set_mean_remainders(mean_remainders)
mean_births: np.ndarray[float] = calculate_mean_during_covid(state.BIRTH_MATRIX, state.WEEKLY_INDEX)
#state.set_no_covid_births(mean_births)
state.set_no_covid_births(state.BIRTH_MATRIX)
mean_deaths: np.ndarray[float] = calculate_mean_during_covid(state.DEATHS_MATRIX, state.WEEKLY_INDEX)
state.set_no_covid_deaths(mean_deaths)
#state.set_no_covid_deaths(state.DEATHS_MATRIX)

In [ ]:
def calculate_mean_s_during_covid(data: np.ndarray[float], index: pd.Series) -> np.ndarray[float]:
    df: pd.DataFrame = pd.DataFrame(data.T, index=index, columns=state.AGE_GROUPS)
    timerange_for_mean = ((df.index > config.GlobalConfig.START_OF_VACCINATION + pd.DateOffset(years=-5))
                         & (df.index < config.GlobalConfig.START_OF_VACCINATION))
    timerange_for_mean = ((df.index > pd.Timestamp(2019,1,1) + pd.DateOffset(years=-5))
                         & (df.index < pd.Timestamp(2019,1,1)))
    mean_data: pd.DataFrame = df[timerange_for_mean].groupby(df.index[timerange_for_mean].map(lambda i: i.week)).mean()
    no_cov_data: np.ndarray = np.zeros((len(state.AGE_GROUPS), len(state.WEEKLY_INDEX)))
    min_t = len(state.WEEKLY_INDEX) - len(index)
    max_t = len(state.WEEKLY_INDEX)
    for t in range(min_t, max_t):
        no_cov_data[:, t] = data[:, t - min_t]
        if (state.WEEKLY_INDEX[t] > pd.Timestamp("2020-03-16")) and (state.WEEKLY_INDEX[t] < pd.Timestamp("2021-09-01")):
            week_of_year: int = state.WEEKLY_INDEX[t].week
            if week_of_year > len(mean_data):
                no_cov_data[:, t] = mean_data.loc[week_of_year-1]
            else:
                no_cov_data[:, t] = mean_data.loc[week_of_year] - state.BASELINE_SCENARIO.V1[:,t-1]
    return no_cov_data

In [ ]:
def calculate_mean_i_during_covid(data: np.ndarray[float], mean_s: np.ndarray[float], index: pd.Series) -> np.ndarray[float]:
    no_cov_data: np.ndarray = np.zeros((len(state.AGE_GROUPS), len(state.WEEKLY_INDEX)))
    for t in range(0, len(state.WEEKLY_INDEX)-1):
        if state.WEEKLY_INDEX[t] < pd.Timestamp("2020-03-16"):
            no_cov_data[:, t] = data[:, t]
        else:
            demographic_changes = state.NO_COVID_BIRTHS[:, t] - state.NO_COVID_DEATHS[:, t] * (mean_s[:,t] / state.NO_COVID_POPULATION[:, t])
            #no_cov_data[:, t] = - mean_s[:,t+1] + mean_s[:,t] + demographic_changes - state.BASELINE_SCENARIO.V1[:,t]
            no_cov_data[:, t] = state.BASELINE_SCENARIO.S[:,t] + state.BIRTH_MATRIX[:,t] - state.DEATHS_MATRIX[:,t] * (state.BASELINE_SCENARIO.S[:,t] / state.POPULATION[:, t]) - state.BASELINE_SCENARIO.V1[:,t] * GlobalConfig.V1_EFFICACY - state.BASELINE_SCENARIO.S[:,t+1]
            if scenarioCalculator.is_time_for_aging(t): scenarioCalculator.handle_aging_of_infectious_cases(no_cov_data, t)
    return no_cov_data

In [ ]:
mean_s: np.ndarray[float] = calculate_mean_s_during_covid(state.BASELINE_SCENARIO.S, state.WEEKLY_INDEX)
#mean_i: np.ndarray[float] = calculate_mean_i_during_covid(state.BASELINE_SCENARIO.I, mean_s, state.WEEKLY_INDEX)

In [ ]:
plotting.plot_model_results_age_range_in_range(mean_s, 0, 4, "mean s")
plotting.plot_model_results_age_range_in_range(mean_s, 5, 9, "mean s")
plotting.plot_model_results_age_range_in_range(mean_s, 10, 12, "mean s")

In [ ]:
plotting.plot_model_results_age_range_in_range(state.NO_COVID_DEATHS, 0, 4, "node covid death cases")
plotting.plot_model_results_age_range_in_range(state.NO_COVID_DEATHS, 5, 9, "node covid death cases")
plotting.plot_model_results_age_range_in_range(state.NO_COVID_DEATHS, 10, 12, "node covid death cases")

In [ ]:
mean_i = calculate_mean_during_covid(state.BASELINE_SCENARIO.I, state.WEEKLY_INDEX)
state.set_mean_i(mean_i)
scenarioCalculator.calculate_number_of_susceptible_cases_in_baseline(False)
state.MEAN_I_SCENARIO.I = mean_i
state.MEAN_I_SCENARIO.compute_i_series()

In [ ]:
plotting.plot_model_results_age_range_in_range(state.BASELINE_SCENARIO.I, 0, 2, "baseline")
plotting.plot_model_results_age_range_in_range(state.MEAN_I_SCENARIO.I, 0, 2, "mean i")
plotting.plot_model_results_age_range_in_range(state.BASELINE_SCENARIO.I-state.MEAN_I_SCENARIO.I, 0, 2, "difference")

In [ ]:
plotting.plot_model_results_age_range_in_range(state.MEAN_I_SCENARIO.S, 0, 4, "no covid baseline")
plotting.plot_model_results_age_range_in_range(state.MEAN_I_SCENARIO.S, 5, 9, "no covid baseline")
plotting.plot_model_results_age_range_in_range(state.MEAN_I_SCENARIO.S, 10, 12, "no covid baseline")

In [ ]:
plotting.plot_model_results_age_range_in_range(state.MEAN_I_SCENARIO.I, 0, 4, "no covid baseline")
plotting.plot_model_results_age_range_in_range(state.MEAN_I_SCENARIO.I, 5, 9, "no covid baseline")
plotting.plot_model_results_age_range_in_range(state.MEAN_I_SCENARIO.I, 10, 12, "no covid baseline")

In [ ]:
plotting.plot_compare_scenario_to_actual_case(state.MEAN_I_SCENARIO.weekly_i_series, state.MEAN_I_SCENARIO.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(state.MEAN_I_SCENARIO.I, 0, 4, state.MEAN_I_SCENARIO.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(state.MEAN_I_SCENARIO.I, 5, 9, state.MEAN_I_SCENARIO.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(state.MEAN_I_SCENARIO.I, 10, 12, state.MEAN_I_SCENARIO.name)

In [ ]:
def cohort_cumulative(df_index, birth_year, infections_matrix, vaccinations_matrix):
    """
    Általános függvény egy születési kohorsz kumulatív fertőzéseinek és oltásainak meghatározására.

    df_index: pandas index (heti idősor)
    birth_year: pl. 2019
    infections_matrix: I[age, time]
    vaccinations_matrix: V[age, time] vagy V1 + V2
    """

    n_time = len(df_index)
    cum_inf = np.zeros(n_time)
    cum_vac = np.zeros(n_time)

    for t in range(n_time):
        date = df_index[t]
        age_group = date.year - birth_year
        if date.month < 9:
            age_group -= 1

        # ha a kohorsz még nem létezik (pl. 2018-ban a 2019-es kohorsz)
        if age_group < 0:
            continue

        # ha túlöregedett a mátrixból (pl. 30 év felett nincs adat)
        if age_group >= infections_matrix.shape[0]:
            break

        # aktuális heti fertőzések és oltások
        weekly_inf = infections_matrix[age_group, t]
        weekly_vac = vaccinations_matrix[age_group, t]

        # kumulálás
        cum_inf[t] = weekly_inf + (cum_inf[t-1] if t > 0 else 0)
        cum_vac[t] = weekly_vac + (cum_vac[t-1] if t > 0 else 0)

    # eredmény DataFrame-ben
    result = pd.DataFrame({
        "cum_infections": cum_inf,
        "cum_vaccinations": cum_vac
    }, index=df_index)

    return result


In [ ]:
import matplotlib.pyplot as plt

mask = state.WEEKLY_INDEX > pd.Timestamp(2019,1 ,25)
cohort_df = cohort_cumulative(state.WEEKLY_INDEX[mask], 2018, state.BASELINE_SCENARIO.I[:,mask], state.BASELINE_SCENARIO.V1[:,mask])

plt.figure(figsize=(10,6))
#plt.plot(cohort_df.index, cohort_df["cum_infections"] + cohort_df["cum_vaccinations"], label="Kumulatív fertőzések")
#plt.plot(cohort_df.index, cohort_df["cum_vaccinations"], label="Kumulatív oltások")
plt.plot(state.WEEKLY_INDEX[mask], state.BASELINE_SCENARIO.S[:2,mask].sum(axis=0), label="BASELINE_SCENARIO.S")
plt.plot(state.WEEKLY_INDEX[mask], scenario_no_vacc.S[:2,mask].sum(axis=0), label="scenario_no_vacc.S")
plt.plot(state.WEEKLY_INDEX[mask], scenario_vacc_level_50.S[:2,mask].sum(axis=0), label="scenario_vacc_level_50.S")

plt.title("összes fogékony")
plt.xlabel("Idő")
plt.ylabel("Fő")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

mask = state.WEEKLY_INDEX > pd.Timestamp(2018,8 ,25)
cohort_df = cohort_cumulative(state.WEEKLY_INDEX[mask], 2018, state.BASELINE_SCENARIO.I[:,mask], state.BASELINE_SCENARIO.V1[:,mask])

plt.figure(figsize=(10,6))
#plt.plot(cohort_df.index, cohort_df["cum_infections"] + cohort_df["cum_vaccinations"], label="Kumulatív fertőzések")
#plt.plot(cohort_df.index, cohort_df["cum_vaccinations"], label="Kumulatív oltások")
plt.plot(state.WEEKLY_INDEX[mask], state.BASELINE_SCENARIO.I[:,mask].sum(axis=0), label="BASELINE_SCENARIO.S")
plt.plot(state.WEEKLY_INDEX[mask], scenario_no_vacc.I[:,mask].sum(axis=0), label="scenario_no_vacc.S")
plt.plot(state.WEEKLY_INDEX[mask], scenario_vacc_level_50.I[:,mask].sum(axis=0), label="scenario_vacc_level_50.S")

plt.title("összes fertőzés")
plt.xlabel("Idő")
plt.ylabel("Fő")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plotting.plot_model_actual_scat_comparison(state.MEAN_I_SCENARIO)

### Calculate and plot remainders without covid

In [ ]:
scenarioCalculator.calculate_remainders(False)

In [ ]:
plotting.plot_model_results_age_range_in_range(state.MEAN_REMAINDERS, 0, 4, "mean remainders")
plotting.plot_model_results_age_range_in_range(state.MEAN_REMAINDERS, 5, 9, "mean remainders")
plotting.plot_model_results_age_range_in_range(state.MEAN_REMAINDERS, 10, 12, "mean remainders")

In [ ]:
plotting.plot_model_results_age_range_in_range(state.REMAINDERS, 0, 4, "remainders")
plotting.plot_model_results_age_range_in_range(state.MEAN_REMAINDERS, 0, 4, "mean remainders")
plotting.plot_model_results_age_range_in_range(state.REMAINDERS, 5, 9, "remainders")
plotting.plot_model_results_age_range_in_range(state.MEAN_REMAINDERS, 5, 9, "mean remainders")
plotting.plot_model_results_age_range_in_range(state.REMAINDERS, 10, 12, "remainders")
plotting.plot_model_results_age_range_in_range(state.MEAN_REMAINDERS, 10, 12, "mean remainders")

## 16. Change vaccination levels

### vaccination level = 1

In [ ]:
no_cov_vacc_level_1 = scenarioCalculator.calculate_number_of_cases_for_scenario(
    #Scenarios.BASELINE, 1, GlobalConfig.START_OF_VACCINATION, True)
    Scenarios.NO_COVID_BASELINE, 1, GlobalConfig.START_OF_VACCINATION, False)
ALL_SCENARIOS.append(no_cov_vacc_level_1)

In [ ]:
plotting.plot_model_actual_scat_comparison(no_cov_vacc_level_1)
plotting.plot_model_results_age_range_in_range(no_cov_vacc_level_1.I, 0, 4, no_cov_vacc_level_1.name)
plotting.plot_model_results_age_range_in_range(no_cov_vacc_level_1.I, 5, 9, no_cov_vacc_level_1.name)
plotting.plot_model_results_age_range_in_range(no_cov_vacc_level_1.I, 10, 12, no_cov_vacc_level_1.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(no_cov_vacc_level_1.weekly_i_series, no_cov_vacc_level_1.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_vacc_level_1.I, 0, 4, no_cov_vacc_level_1.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_vacc_level_1.I, 5, 9, no_cov_vacc_level_1.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_vacc_level_1.I, 10, 12, no_cov_vacc_level_1.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(state.MEAN_I_SCENARIO.I - no_cov_vacc_level_1.I, 0, 4, no_cov_vacc_level_1.name)
plotting.plot_model_results_age_range_in_range(state.MEAN_I_SCENARIO.I - no_cov_vacc_level_1.I, 5, 9, no_cov_vacc_level_1.name)
plotting.plot_model_results_age_range_in_range(state.MEAN_I_SCENARIO.I - no_cov_vacc_level_1.I, 10, 12, no_cov_vacc_level_1.name)

### vaccination level = 0.75

In [ ]:
no_cov_vacc_level_75 = scenarioCalculator.calculate_number_of_cases_for_scenario(
    Scenarios.NO_COVID_VACC_LEVEL_75, 0.75, GlobalConfig.START_OF_VACCINATION, False)
ALL_SCENARIOS.append(no_cov_vacc_level_75)

In [ ]:
plotting.plot_model_actual_scat_comparison(no_cov_vacc_level_75)
plotting.plot_model_results_age_range_in_range(no_cov_vacc_level_75.I, 0, 4, no_cov_vacc_level_75.name)
plotting.plot_model_results_age_range_in_range(no_cov_vacc_level_75.I, 5, 9, no_cov_vacc_level_75.name)
plotting.plot_model_results_age_range_in_range(no_cov_vacc_level_75.I, 10, 12, no_cov_vacc_level_75.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(no_cov_vacc_level_75.weekly_i_series, no_cov_vacc_level_75.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_vacc_level_75.I, 0, 4, no_cov_vacc_level_75.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_vacc_level_75.I, 5, 9, no_cov_vacc_level_75.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_vacc_level_75.I, 10, 12, no_cov_vacc_level_75.name)

### vaccination level = 0.5

In [ ]:
no_cov_vacc_level_50 = scenarioCalculator.calculate_number_of_cases_for_scenario(
    Scenarios.NO_COVID_VACC_LEVEL_50, 0.5, GlobalConfig.START_OF_VACCINATION, False)
ALL_SCENARIOS.append(no_cov_vacc_level_50)

In [ ]:
plotting.plot_model_actual_scat_comparison(no_cov_vacc_level_50)
plotting.plot_model_results_age_range_in_range(no_cov_vacc_level_50.I, 0, 4, no_cov_vacc_level_50.name)
plotting.plot_model_results_age_range_in_range(no_cov_vacc_level_50.I, 5, 9, no_cov_vacc_level_50.name)
plotting.plot_model_results_age_range_in_range(no_cov_vacc_level_50.I, 10, 12, no_cov_vacc_level_50.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(no_cov_vacc_level_50.weekly_i_series, no_cov_vacc_level_50.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_vacc_level_50.I, 0, 4, no_cov_vacc_level_50.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_vacc_level_50.I, 5, 9, no_cov_vacc_level_50.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_vacc_level_50.I, 10, 12, no_cov_vacc_level_50.name)

### vaccination level = 0.25

In [ ]:
no_cov_vacc_level_25 = scenarioCalculator.calculate_number_of_cases_for_scenario(
    Scenarios.NO_COVID_VACC_LEVEL_25, 0.25, GlobalConfig.START_OF_VACCINATION, False)
ALL_SCENARIOS.append(no_cov_vacc_level_25)

In [ ]:
plotting.plot_model_actual_scat_comparison(no_cov_vacc_level_25)
plotting.plot_model_results_age_range_in_range(no_cov_vacc_level_25.I, 0, 4, no_cov_vacc_level_25.name)
plotting.plot_model_results_age_range_in_range(no_cov_vacc_level_25.I, 5, 9, no_cov_vacc_level_25.name)
plotting.plot_model_results_age_range_in_range(no_cov_vacc_level_25.I, 10, 12, no_cov_vacc_level_25.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(no_cov_vacc_level_25.weekly_i_series, no_cov_vacc_level_25.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_vacc_level_25.I, 0, 4, no_cov_vacc_level_25.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_vacc_level_25.I, 5, 9, no_cov_vacc_level_25.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_vacc_level_25.I, 10, 12, no_cov_vacc_level_25.name)

### vaccination level = 0

In [ ]:
no_cov_no_vacc = scenarioCalculator.calculate_number_of_cases_for_scenario(
    Scenarios.NO_COVID_NO_VACC, 0, GlobalConfig.START_OF_VACCINATION, False)
ALL_SCENARIOS.append(no_cov_no_vacc)

In [ ]:
plotting.plot_model_actual_scat_comparison(no_cov_no_vacc)
plotting.plot_model_results_age_range_in_range(no_cov_no_vacc.I, 0, 4, no_cov_no_vacc.name)
plotting.plot_model_results_age_range_in_range(no_cov_no_vacc.I, 5, 9, no_cov_no_vacc.name)
plotting.plot_model_results_age_range_in_range(no_cov_no_vacc.I, 10, 12, no_cov_no_vacc.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(no_cov_no_vacc.weekly_i_series, no_cov_no_vacc.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_no_vacc.I, 0, 4, no_cov_no_vacc.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_no_vacc.I, 5, 9, no_cov_no_vacc.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_no_vacc.I, 10, 12, no_cov_no_vacc.name)

## 17. Starting vaccination earlier

### Start vaccination 1 year earlier (2018.09.01)

In [ ]:
no_cov_1_year_minus = scenarioCalculator.calculate_number_of_cases_for_scenario(
    Scenarios.NO_COVID_VACC_1_YEAR_MINUS, 1, GlobalConfig.START_OF_VACCINATION + pd.DateOffset(years=-1), False)
ALL_SCENARIOS.append(no_cov_1_year_minus)

In [ ]:
plotting.plot_model_actual_scat_comparison(no_cov_1_year_minus)
plotting.plot_model_results_age_range_in_range(no_cov_1_year_minus.I, 0, 4, no_cov_1_year_minus.name)
plotting.plot_model_results_age_range_in_range(no_cov_1_year_minus.I, 5, 9, no_cov_1_year_minus.name)
plotting.plot_model_results_age_range_in_range(no_cov_1_year_minus.I, 10, 12, no_cov_1_year_minus.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(no_cov_1_year_minus.weekly_i_series, no_cov_1_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_1_year_minus.I, 0, 4, no_cov_1_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_1_year_minus.I, 5, 9, no_cov_1_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_1_year_minus.I, 10, 12, no_cov_1_year_minus.name)

### Start vaccination 2 years earlier (2017.09.01)

In [ ]:
no_cov_2_year_minus = scenarioCalculator.calculate_number_of_cases_for_scenario(
    Scenarios.NO_COVID_VACC_2_YEAR_MINUS, 1, GlobalConfig.START_OF_VACCINATION + pd.DateOffset(years=-2), False)
ALL_SCENARIOS.append(no_cov_2_year_minus)

In [ ]:
plotting.plot_model_actual_scat_comparison(no_cov_2_year_minus)
plotting.plot_model_results_age_range_in_range(no_cov_2_year_minus.I, 0, 4, no_cov_2_year_minus.name)
plotting.plot_model_results_age_range_in_range(no_cov_2_year_minus.I, 5, 9, no_cov_2_year_minus.name)
plotting.plot_model_results_age_range_in_range(no_cov_2_year_minus.I, 10, 12, no_cov_2_year_minus.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(no_cov_2_year_minus.weekly_i_series, no_cov_2_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_2_year_minus.I, 0, 4, no_cov_2_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_2_year_minus.I, 5, 9, no_cov_2_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_2_year_minus.I, 10, 12, no_cov_2_year_minus.name)

### Start vaccination 3 years earlier (2016.09.01)

In [ ]:
no_cov_3_year_minus = scenarioCalculator.calculate_number_of_cases_for_scenario(
    Scenarios.NO_COVID_VACC_3_YEAR_MINUS, 1, GlobalConfig.START_OF_VACCINATION + pd.DateOffset(years=-3), False)
ALL_SCENARIOS.append(no_cov_3_year_minus)

In [ ]:
plotting.plot_model_actual_scat_comparison(no_cov_3_year_minus)
plotting.plot_model_results_age_range_in_range(no_cov_3_year_minus.I, 0, 4, no_cov_3_year_minus.name)
plotting.plot_model_results_age_range_in_range(no_cov_3_year_minus.I, 5, 9, no_cov_3_year_minus.name)
plotting.plot_model_results_age_range_in_range(no_cov_3_year_minus.I, 10, 12, no_cov_3_year_minus.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(no_cov_3_year_minus.I, 1, 5, no_cov_3_year_minus.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(no_cov_3_year_minus.I, 3, 6, no_cov_3_year_minus.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(no_cov_3_year_minus.V1, 0, 1, no_cov_3_year_minus.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(no_cov_3_year_minus.weekly_i_series, no_cov_3_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_3_year_minus.I, 0, 4, no_cov_3_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_3_year_minus.I, 5, 9, no_cov_3_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_3_year_minus.I, 10, 12, no_cov_3_year_minus.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(no_cov_3_year_minus.S, 0, 4, no_cov_3_year_minus.name)
plotting.plot_model_results_age_range_in_range(no_cov_3_year_minus.S, 5, 9, no_cov_3_year_minus.name)
plotting.plot_model_results_age_range_in_range(no_cov_3_year_minus.S, 10, 12, no_cov_3_year_minus.name)

### Start vaccination 5 years earlier (2014.09.01)

In [ ]:
no_cov_5_year_minus = scenarioCalculator.calculate_number_of_cases_for_scenario(
    Scenarios.NO_COVID_VACC_5_YEAR_MINUS, 1, GlobalConfig.START_OF_VACCINATION + pd.DateOffset(years=-5), False)
ALL_SCENARIOS.append(no_cov_5_year_minus)

In [ ]:
plotting.plot_model_actual_scat_comparison(no_cov_5_year_minus)
plotting.plot_model_results_age_range_in_range(no_cov_5_year_minus.I, 0, 4, no_cov_5_year_minus.name)
plotting.plot_model_results_age_range_in_range(no_cov_5_year_minus.I, 5, 9, no_cov_5_year_minus.name)
plotting.plot_model_results_age_range_in_range(no_cov_5_year_minus.I, 10, 12, no_cov_5_year_minus.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(no_cov_5_year_minus.S, 0, 4, no_cov_5_year_minus.name)
plotting.plot_model_results_age_range_in_range(no_cov_5_year_minus.S, 5, 9, no_cov_5_year_minus.name)
plotting.plot_model_results_age_range_in_range(no_cov_5_year_minus.S, 10, 12, no_cov_5_year_minus.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(no_cov_5_year_minus.V1, 0, 2, no_cov_5_year_minus.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(no_cov_5_year_minus.weekly_i_series, no_cov_5_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_5_year_minus.I, 0, 4, no_cov_5_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_5_year_minus.I, 5, 9, no_cov_5_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_5_year_minus.I, 10, 12, no_cov_5_year_minus.name)

### Start vaccination 8 year earlier (2011.09.01)

In [ ]:
no_cov_8_year_minus = scenarioCalculator.calculate_number_of_cases_for_scenario(
    Scenarios.NO_COVID_VACC_8_YEAR_MINUS, 1, GlobalConfig.START_OF_VACCINATION + pd.DateOffset(years=-8), False)
ALL_SCENARIOS.append(no_cov_8_year_minus)

In [ ]:
plotting.plot_model_actual_scat_comparison(no_cov_8_year_minus)
plotting.plot_model_results_age_range_in_range(no_cov_8_year_minus.I, 0, 4, no_cov_8_year_minus.name)
plotting.plot_model_results_age_range_in_range(no_cov_8_year_minus.I, 5, 9, no_cov_8_year_minus.name)
plotting.plot_model_results_age_range_in_range(no_cov_8_year_minus.I, 10, 12, no_cov_8_year_minus.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(no_cov_8_year_minus.weekly_i_series, no_cov_8_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_8_year_minus.I, 0, 4, no_cov_8_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_8_year_minus.I, 5, 9, no_cov_8_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_8_year_minus.I, 10, 12, no_cov_8_year_minus.name)

In [ ]:
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_8_year_minus.S, 0, 4, no_cov_8_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_8_year_minus.S, 5, 9, no_cov_8_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_8_year_minus.S, 10, 12, no_cov_8_year_minus.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(no_cov_8_year_minus.S, 0, 4, no_cov_8_year_minus.name)
plotting.plot_model_results_age_range_in_range(no_cov_8_year_minus.S, 5, 9, no_cov_8_year_minus.name)
plotting.plot_model_results_age_range_in_range(no_cov_8_year_minus.S, 10, 12, no_cov_8_year_minus.name)

### Start vaccination 13 years earlier (2006.09.01)

In [ ]:
no_cov_13_year_minus = scenarioCalculator.calculate_number_of_cases_for_scenario(
    Scenarios.NO_COVID_VACC_13_YEAR_MINUS, 1, GlobalConfig.START_OF_VACCINATION + pd.DateOffset(years=-13), False)
ALL_SCENARIOS.append(no_cov_13_year_minus)

In [ ]:
plotting.plot_model_actual_scat_comparison(no_cov_13_year_minus)
plotting.plot_model_results_age_range_in_range(no_cov_13_year_minus.I, 0, 4, no_cov_13_year_minus.name)
plotting.plot_model_results_age_range_in_range(no_cov_13_year_minus.I, 5, 9, no_cov_13_year_minus.name)
plotting.plot_model_results_age_range_in_range(no_cov_13_year_minus.I, 10, 12, no_cov_13_year_minus.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(no_cov_13_year_minus.weekly_i_series, no_cov_13_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_13_year_minus.I, 0, 4, no_cov_13_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_13_year_minus.I, 5, 9, no_cov_13_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_13_year_minus.I, 10, 12, no_cov_13_year_minus.name)

In [ ]:
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_13_year_minus.V1, 0, 1, no_cov_13_year_minus.name)

In [ ]:
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_13_year_minus.S, 0, 4, no_cov_13_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_13_year_minus.S, 5, 9, no_cov_13_year_minus.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(no_cov_13_year_minus.S, 10, 12, no_cov_13_year_minus.name)

In [ ]:
plotting.plot_annual_annual_number_of_cases_for_age_groups(state.BASELINE_SCENARIO.I, 0, 4, state.BASELINE_SCENARIO.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(state.BASELINE_SCENARIO.I, 5, 9, state.BASELINE_SCENARIO.name)
plotting.plot_annual_annual_number_of_cases_for_age_groups(state.BASELINE_SCENARIO.I, 10, 12, state.BASELINE_SCENARIO.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(no_cov_13_year_minus.S, 0, 4, no_cov_13_year_minus.name)
plotting.plot_model_results_age_range_in_range(no_cov_13_year_minus.S, 5, 9, no_cov_13_year_minus.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(state.BASELINE_SCENARIO.S, 0, 4, no_cov_13_year_minus.name)
plotting.plot_model_results_age_range_in_range(state.BASELINE_SCENARIO.S, 5, 9, no_cov_13_year_minus.name)

In [ ]:
japan = scenarioCalculator.calculate_number_of_cases_for_scenario("like in Japan", 0.8, GlobalConfig.START_OF_VACCINATION + pd.DateOffset(-5), True)

In [ ]:
plotting.plot_model_actual_scat_comparison(japan)
plotting.plot_model_results_age_range_in_range(japan.I, 0, 4, "like in Japan")
plotting.plot_model_results_age_range_in_range(japan.I, 5, 9, "like in Japan")
plotting.plot_model_results_age_range_in_range(japan.I, 10, 12, "like in Japan")